In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.chdir('/content/drive/MyDrive/images')

In [ ]:
!pip install opencv-python face_recognition

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 8.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for face-recognition-models: filename=face_recognition_models-0.3.0-py2.py3-none-any.whl size=100566164 sha256=4e4ca75852a8d88b86c28038638142b8dff5c846865d57a39ccfef3906615fa2
  Stored in directory: /root/.cache/pip/wheels/7a/eb/cf/e9eced74122b679557f597bb7c8e4c739cfcac526db1fd523d
Successfully built face-recognition-models


In [ ]:
!pip install face_recognition

In [ ]:
import cv2
import face_recognition
import numpy as np

In [ ]:
import glob
known_images = []
known_names = []

for image_path in glob.glob("images/*.jpg"):
    image = cv2.imread(image_path)
    face_locations = face_recognition.face_locations(image)
    face_encodings = face_recognition.face_encodings(image, face_locations)

    for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):

        known_images.append(face_encoding)
        known_names.append(image_path.split("/")[-2])

In [ ]:
video_capture = cv2.VideoCapture(0)

In [ ]:
from google.colab.patches import cv2_imshow

In [ ]:
while True:
    ret, frame = video_capture.read()
    cv2_imshow(frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break


AttributeError: 'NoneType' object has no attribute 'clip'

In [ ]:
while True:
    ret, frame = video_capture.read()

    # Resize the frame for faster processing
    small_frame = cv2.resize(frame, None, fx=0.25, fy=0.25)

    # Convert the image from BGR color space to RGB
    rgb_small_frame = small_frame[:, :, ::-1]

    # Find all faces in the current frame
    face_locations = face_recognition.face_locations(rgb_small_frame)
    face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)


    # Loop through each face found in the current frame
    for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
        matches = face_recognition.compare_faces(known_images, face_encoding)
        name = "Unknown"

        # Find the closest match
        face_distances = face_recognition.face_distance(known_images, face_encoding)
        best_match_index = np.argmin(face_distances)
        if matches[best_match_index]:
            name = known_names[best_match_index]

        # Mark the detected face with a rectangle and label
        cv2.rectangle(frame, (left * 4, top * 4), (right * 4, bottom * 4), (0, 0, 255), 2)
        cv2.rectangle(frame, (left * 4, bottom * 4 - 35), (right * 4, bottom * 4), (0, 0, 255), cv2.FILLED)

        cv2.putText(frame, name, (left * 4 + 6, bottom * 4 - 6), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)

    # Display the resulting image
    cv2.imshow('Video', frame)

    # Quit the program if the user presses 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release the video capture and close all windows
video_capture.release()
cv2.destroyAllWindows()


error: OpenCV(4.10.0) /io/opencv/modules/imgproc/src/resize.cpp:4152: error: (-215:Assertion failed) !ssize.empty() in function 'resize'
